# **Start Section:**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Uninstall existing scikit-learn to avoid conflicts
!pip uninstall -y scikit-learn
# Install specific versions of libraries to avoid conflicts
!pip install scikit-learn==1.5.2
!pip install bayesian-optimization
!pip install optuna
!pip install gpboost
!pip install shap
!pip install ngboost
!pip install dask[dataframe]
!pip install torch seaborn
!pip install lightgbm
!pip install xgboost
!pip install lime
!pip install interpret
!pip install optunahub
!pip install cmaes
!pip install plotly kaleido
!pip install openpyxl
!pip install -U kaleido
!pip install properscoring
!pip install XlsxWriter
!pip install cython
!pip install pgbm
!pip install torch
!pip install cp
!pip install mapie
!pip install torch skorch puncc
# Reinstall scikit-learn to the version required by ngboost
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.6.1
# Reinstall numpy first
!pip install numpy==1.26.4  # Use the version compatible with catboost
# Reinstall catboost
!pip install catboost

Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
  Using cached scikit_learn-1.5.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
Using cached scikit_learn-1.5.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.3 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ngboost 0.5.5 requires scikit-learn<2.0,>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
  Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.

In [ ]:
# Restart the runtime to apply changes
import os
os._exit(00)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage

In [4]:
train_data_path = "./drive/MyDrive/Concrete_ml/train.csv"
test_data_path = "./drive/MyDrive/Concrete_ml/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [5]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [6]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [7]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**

In [40]:
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']


In [41]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'GBM': 'Gradient Boosting',
        'CatBoost': 'CatBoost',
        'NGBoost': 'NGBoost',
        'HGBR' : 'HistGradientBoosting',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [42]:
def quantile_regression_print(predictions_df, file_path):
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label='Actual Concrete Strength', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel('Concrete Strength')
    plt.title('Actual vs. Predicted Quantile Strengths')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label='Actual Strength', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(' Concrete Strength')
    plt.title('Actual Strength with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    wb.save(file_path)

In [43]:
def evaluation_features_print(predictions_df, X_test, file_path):
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=[f'Feature_{i}' for i in range(X_test.shape[1])])

    # Rename columns to match feature_names
    X_test.columns = feature_names

    # Get quantile columns from predictions_df
    # Assuming that quantile columns are all columns except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']
    quantiles = quantile_columns  # Assuming the column names represent quantiles

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        # Load the existing workbook
        wb = load_workbook(file_path)
        # Create a new worksheet
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        # Create a new workbook and add a worksheet
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Start row for placing images

    # Loop through each feature and create a plot
    for feature_to_plot in feature_names:
        # Extract feature values corresponding to the test set
        feature_values = X_test[feature_to_plot]

        # Create a DataFrame for Plotly
        plotly_df = predictions_df.copy()
        plotly_df[feature_to_plot] = feature_values.values

        # Melt the DataFrame to long format
        melted_df = plotly_df.melt(
            id_vars=[feature_to_plot, 'Actual'],
            value_vars=quantiles,
            var_name='Quantile',
            value_name='Prediction'
        )

        # Optional: Sort the data for better visualization
        melted_df.sort_values(by=feature_to_plot, inplace=True)

        # Plot
        fig = px.line(
            melted_df,
            x=feature_to_plot,
            y='Prediction',
            color='Quantile',
            title=f'Predicted Quantiles vs. {feature_to_plot}',
            labels={'Prediction': 'Strength'}
        )

        # Add actual values as scatter points
        fig.add_scatter(
            x=plotly_df[feature_to_plot],
            y=plotly_df['Actual'],
            mode='markers',
            name='Actual Strength'
        )

        # Save the plot to a BytesIO object
        img_data = BytesIO()
        fig.write_image(img_data, format='png')
        img_data.seek(0)

        # Insert the image into the Excel sheet
        img = Image(img_data)
        img.anchor = f'A{image_row}'
        ws.add_image(img)

        # Increase image_row to place the next image below this one
        image_row += 20  # Adjust as needed to avoid overlap

    # Save the workbook
    wb.save(file_path)


In [44]:
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(filename=excel_file_path)

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)//2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    crps_values = ps.crps_ensemble(y_true, ensemble_predictions)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    wb.save(excel_file_path)

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [45]:
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel('Concrete Strength')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    plt.savefig(image_path)
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(excel_file_path)

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    workbook.save(excel_file_path)

In [55]:
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Plotly
    fig = go.Figure()

    # Plot Actual Values
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=actual_values,
        mode='lines',
        name='Actual',
        line=dict(color='black', width=2)
    ))

    # Plot Mean Prediction
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=mean_prediction,
        mode='lines',
        name='Mean Prediction',
        line=dict(color='darkorange', width=2)
    ))

    # Plot Weighted Prediction Intervals
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=lower_bound,
        fill=None,
        mode='lines',
        line=dict(color='blue', width=0),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=upper_bound,
        fill='tonexty',
        mode='lines',
        line=dict(color='blue', width=0),
        name='Weighted Interval',
        fillcolor='rgba(0, 0, 255, 0.3)'
    ))

    # Plot Naive Prediction Intervals
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=naive_lower_bound,
        fill=None,
        mode='lines',
        line=dict(color='red', width=0),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=predictions_df.index,
        y=naive_upper_bound,
        fill='tonexty',
        mode='lines',
        line=dict(color='red', width=0),
        name='Naive Interval',
        fillcolor='rgba(255, 0, 0, 0.3)'
    ))

    # Update layout for better visualization
    fig.update_layout(
        title='Quantile Prediction Intervals',
        xaxis_title='Sample Number',
        yaxis_title='Concrete Strength',
        xaxis=dict(
            showline=True,
            showgrid=False,
            showticklabels=True,
            linecolor='black',
            linewidth=2,
            ticks='outside',
            tickfont=dict(
                family='Arial',
                size=12,
                color='black',
            ),
        ),
        yaxis=dict(
            showline=True,
            showgrid=False,
            showticklabels=True,
            linecolor='black',
            linewidth=2,
            ticks='outside',
            tickfont=dict(
                family='Arial',
                size=12,
                color='black',
            ),
        ),
        legend=dict(
            x=1.05,
            y=1,
            xanchor='left',
            yanchor='top',
            traceorder='normal',
            font=dict(
                family='Arial',
                size=12,
                color='black',
            ),
            bgcolor='rgba(255, 255, 255, 0)',
            bordercolor='black',
            borderwidth=1
        ),
        plot_bgcolor='white',
        width=800,   # Plot width
        height=400,  # Plot height
        margin=dict(
            l=60,    # left margin
            r=150,   # increased right margin to accommodate legend
            b=50,    # bottom margin
            t=50     # top margin
        ),
    )

    # Save the plot to a BytesIO object with adjusted dimensions
    img_data = BytesIO()
    fig.write_image(img_data, format='png', width=800, height=400, scale=1)
    img_data.seek(0)

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        # Load the existing workbook
        wb = load_workbook(file_path)
        # Create a new worksheet
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        # Create a new workbook and add a worksheet
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    # Insert the image into the Excel sheet
    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)

    # Adjust the column widths to accommodate the image
    ws.column_dimensions['A'].width = 30

    # Save the workbook
    wb.save(file_path)

# **Hyperparameter Tuning using Autosampler Optuna**

In [19]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Define objective and metric functions for PGBM
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss


def hyperparameter_tuning_all(X_train, y_train, X_test, y_test):

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)


    models = {
        'Gradient Boosting': (GradientBoostingRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 10],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': [1, 2, 3, 4, 5]
        }),
        'XGBoost': (XGBRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'min_child_weight': [1, 3, 5],
            'gamma': [0, 0.1, 0.5, 1],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9]
        }),
        'LightGBM': (LGBMRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'verbose': [-1]
        }),
        'GPBoost': (GPBoostRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'n_estimators': [100, 200, 300, 400, 500],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'alpha': [0.1, 0.5, 1.0],
            'lambda': [0.1, 0.5, 1.0],
            'verbose': [-1]
        }),
        'CatBoost': (CatBoostRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'iterations': [100, 200, 300, 400, 500],
            'depth': [3, 5, 7],
            'l2_leaf_reg': [1, 3, 5],
            'border_count': [32, 64, 128],
            'verbose': [0]
        }),
        'NGBoost': (NGBRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'natural_gradient': [True, False],
            'minibatch_frac': [0.5, 0.7],
            'verbose': [0]
        }),
        'HistGradientBoosting': (HistGradientBoostingRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_iter': [100, 200, 300, 400, 500],
            'max_depth': [3, 5, 7, None],
            'min_samples_leaf': [1, 5, 10],
            'max_leaf_nodes': [15, 31, 63, None],
            'l2_regularization': [0.0, 0.1, 0.5, 1.0],
            'max_bins': [64, 128, 255],
            'early_stopping': [True, False]
        }),
        'PGBM': (PGBM, {}) # PGBM is handled separately
    }

    pruners = [
        optuna.pruners.MedianPruner(),
        optuna.pruners.NopPruner(),
        optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=3),
        optuna.pruners.PercentilePruner(25.0),
        optuna.pruners.SuccessiveHalvingPruner(),
        optuna.pruners.HyperbandPruner(),
        optuna.pruners.ThresholdPruner(lower=0.1),
        optuna.pruners.WilcoxonPruner()
    ]

    best_scores = {}

    for model_name, (model_class, param_space) in models.items():
        for pruner in pruners:
            pruner_name = pruner.__class__.__name__
            print(f"Running Optuna for {model_name} with {pruner_name}...")
            try:
                sampler = optunahub.load_module("samplers/auto_sampler").AutoSampler()
                study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)

                if model_name == 'PGBM':
                    def pgbm_objective(trial):
                        params = {
                            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
                            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
                            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
                            'max_leaves': trial.suggest_int('max_leaves', 15, 63),
                            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
                            'verbose': 0,
                            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
                            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
                            'tree_correlation': trial.suggest_float('tree_correlation', 0.0, 0.5)
                        }

                        model = PGBM()


                        model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=params)
                        y_pred = model.predict(X_test)
                        mse = mean_squared_error(y_test, y_pred)
                        return mse

                    study.optimize(pgbm_objective, n_trials=50)
                else:
                    def objective(trial, model_name, model_class, param_space):
                        params = {}
                        for key, values in param_space.items():
                            if isinstance(values, list):
                                params[key] = trial.suggest_categorical(key, values)
                            elif isinstance(values, tuple):
                                if len(values) == 2:
                                    params[key] = trial.suggest_float(key, values[0], values[1])
                                elif len(values) == 3 and isinstance(values[2], bool) and values[2]:
                                    params[key] = trial.suggest_int(key, values[0], values[1])
                                else:
                                    raise ValueError(f"Invalid parameter range for {key}")

                        model = model_class(**params)

                        model.fit(X_train, y_train)
                        y_pred = model.predict(X_test)
                        mse = mean_squared_error(y_test, y_pred)
                        return mse

                    study.optimize(lambda trial: objective(trial, model_name, model_class, param_space), n_trials=50)

                best_params = study.best_params
                best_model = model_class(**best_params) if model_name != 'PGBM' else PGBM()
                if model_name == 'PGBM':
                    best_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params,)
                else:
                    best_model.fit(X_train, y_train)
                y_pred = best_model.predict(X_test)

                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                corr_coef = np.corrcoef(y_test, y_pred)[0, 1]

                best_scores[(model_name, pruner_name)] = {
                    'best_score': mse,
                    'best_params': best_params,
                    'test_mse': mse,
                    'test_rmse': rmse,
                    'test_corr_coef': corr_coef,
                    'pruner': pruner_name
                }
                print(f"Best MSE for {model_name} with {pruner_name}: {mse} with params: {best_params}")
                print(f"Best RMSE for {model_name} with {pruner_name}: {rmse}")
                print(f"Correlation Coefficient for {model_name} with {pruner_name}: {corr_coef}")
            except Exception as e:
                print(f"Failed to run Optuna for {model_name} with {pruner_name}. Error: {e}")

    if best_scores:
        best_model_name, best_pruner_name = min(best_scores, key=lambda k: best_scores[k]['test_mse'])
        best_model_info = best_scores[(best_model_name, best_pruner_name)]
        print(f"\nBest model on test data: {best_model_name} with {best_pruner_name}")
        print(f"Test MSE: {best_model_info['test_mse']}")
        print(f"Test RMSE: {best_model_info['test_rmse']}")
        print(f"Correlation Coefficient: {best_model_info['test_corr_coef']}")
        print(f"Best Parameters: {best_model_info['best_params']}")
        print(f"Pruner Used: {best_model_info['pruner']}")
    else:
        print("No valid model configurations found.")

    return best_scores

# Example usage
best_scores_autosampler = hyperparameter_tuning_all(X_train, y_train, X_test, y_test)

Running Optuna for Gradient Boosting with MedianPruner...
Best MSE for Gradient Boosting with MedianPruner: 8.374683972547082 with params: {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 2}
Best RMSE for Gradient Boosting with MedianPruner: 2.893904623954819
Correlation Coefficient for Gradient Boosting with MedianPruner: 0.9572851785747521
Running Optuna for Gradient Boosting with NopPruner...
Best MSE for Gradient Boosting with NopPruner: 8.900181176790479 with params: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 3}
Best RMSE for Gradient Boosting with NopPruner: 2.983317143179799
Correlation Coefficient for Gradient Boosting with NopPruner: 0.9546782004278249
Running Optuna for Gradient Boosting with PatientPruner...
Best MSE for Gradient Boosting with PatientPruner: 8.746330567929913 with params: {'n_estimators': 300, 'learning_

In [ ]:
best_scores_autosampler

{('Gradient Boosting', 'MedianPruner'): {'best_score': 10.118895742463085,
  'best_params': {'n_estimators': 500,
   'learning_rate': 0.15,
   'max_depth': 5,
   'min_samples_split': 2,
   'min_samples_leaf': 4,
   'max_features': 2},
  'test_mse': 10.118895742463085,
  'test_rmse': 3.1810211791912177,
  'test_corr_coef': 0.9479964352985609,
  'pruner': 'MedianPruner'},
 ('Gradient Boosting', 'NopPruner'): {'best_score': 8.719154535270686,
  'best_params': {'n_estimators': 300,
   'learning_rate': 0.05,
   'max_depth': 5,
   'min_samples_split': 10,
   'min_samples_leaf': 1,
   'max_features': 5},
  'test_mse': 8.719154535270686,
  'test_rmse': 2.9528214533341983,
  'test_corr_coef': 0.9556373707211432,
  'pruner': 'NopPruner'},
 ('Gradient Boosting', 'PatientPruner'): {'best_score': 9.076561624510754,
  'best_params': {'n_estimators': 300,
   'learning_rate': 0.1,
   'max_depth': 5,
   'min_samples_split': 10,
   'min_samples_leaf': 2,
   'max_features': 2},
  'test_mse': 9.0765616245

# **Conformal Predictions with Lightgbm**

In [56]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  26.693332  28.573971  31.640630  31.189738  31.324406  32.782719   
1  29.940465  31.996373  32.870810  32.472396  34.096739  36.957128   
2  32.809426  33.249756  33.773434  33.157586  31.628715  33.433446   
3  31.609382  33.305136  33.108953  32.061176  33.750094  36.640884   
4  39.991140  40.515910  42.466540  41.791791  40.645095  39.469960   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.219428  32.649583  32.757093  33.738667  34.134266  35.008787   
1  36.287200  36.370060  36.165859  37.553035  36.392291  36.722219   
2  35.014158  34.980493  34.615684  35.602714  35.005381  35.978385   
3  36.069387  36.173710  36.425075  37.032481  36.831628  36.711082   
4  41.651530  42.634563  45.063739  50.632540  50.870090  57.244711   

        0.95  Actual  
0  40.035519    34.0  
1  36.907705    36.0  
2  38.920611    36.0  
3  36.852352    35.0  
4  60.336328    44.6  


In [57]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/LightGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 58.95%


In [58]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/LightGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Concrete_ml/Quantile Regression/LightGBM.xlsx", model_name='LightGBM')

{'empirical_quantiles': [0.16842105263157894,
  0.21052631578947367,
  0.2631578947368421,
  0.28421052631578947,
  0.3473684210526316,
  0.49473684210526314,
  0.5052631578947369,
  0.5684210526315789,
  0.631578947368421,
  0.7052631578947368,
  0.7157894736842105,
  0.7894736842105263,
  0.7578947368421053],
 'average_interval_lengths': [3.1307553887401234,
  5.199864321030318,
  7.7471193108139795],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 15.31983925252315,
 'average_crps': 1.6918562160369908,
 'nll_values': array([ 2.41544267e+00,  1.67855365e+00,  1.67907539e+00,  1.61014061e+00,
         2.85463047e+00,  2.11214677e+00,  5.92533436e+00,  1.32845748e+00,
         2.87828719e+00,  1.81010764e+00,  1.29918090e+00, -1.01145555e-01,
         1.51858653e+00,  2.88267476e+00,  2.93925073e+00,  1.48297463e+00,
         3.00478042e+00,  1.05665877e+00,  2.27716134e+00,  1.26354058e+00,
         2.43591825e+00,  1.77929391e+00,  1.88539441e+00,  2.65313863e+00,
         1.468

In [60]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Concrete_ml/Quantile Regression/LightGBM.xlsx")

The empirical coverage before calibration is: 54.17%
The empirical coverage after calibration is: 93.75%


In [61]:
plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/LightGBM.xlsx")

# **Conformal Predictions with XGBoost**

In [62]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  27.796701  29.969088  30.502728  31.664288  32.108719  32.762737   
1  31.462984  31.633022  34.438965  33.874245  35.450466  35.763268   
2  33.259014  34.097534  34.722240  34.599762  34.497093  34.626606   
3  26.778782  31.111389  32.781166  33.799301  34.428680  34.229195   
4  36.103474  43.170681  42.445774  46.890354  47.703808  43.014904   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.738441  32.996071  33.012646  35.125076  36.123791  38.917351   
1  35.914204  36.539364  36.784740  37.552399  37.287846  37.641369   
2  34.580547  34.897804  34.950493  35.901695  37.315880  40.100983   
3  34.977581  35.857273  35.921989  36.611359  36.422745  37.735302   
4  45.414860  46.675888  44.065041  46.686520  46.776432  44.773247   

        0.95  Actual  
0  41.622665    34.0  
1  38.284386    36.0  
2  42.959587    36.0  
3  38.226135    35.0  
4  51.221287    44.6  


In [63]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/XGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 73.68%


In [64]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/XGBoost.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Concrete_ml/Quantile Regression/XGBoost.xlsx", model_name='XGBoost')

{'empirical_quantiles': [0.11578947368421053,
  0.15789473684210525,
  0.21052631578947367,
  0.25263157894736843,
  0.3684210526315789,
  0.4,
  0.47368421052631576,
  0.5368421052631579,
  0.6105263157894737,
  0.6736842105263158,
  0.7052631578947368,
  0.7263157894736842,
  0.8526315789473684],
 'average_interval_lengths': [3.5351017, 6.5463953, 11.524878],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 2.455764036883438,
 'average_crps': 1.873903961636505,
 'nll_values': array([2.3997396 , 1.64906088, 2.11618434, 2.16591933, 2.45973445,
        2.37393046, 3.27704001, 1.00159014, 3.23590837, 2.34223112,
        0.65319909, 2.28571662, 2.20948081, 3.03149705, 3.09470297,
        2.14809003, 3.06943499, 2.00427479, 2.83292852, 1.51385156,
        2.5560902 , 1.89941256, 2.17527177, 1.91616496, 1.69929933,
        1.52320072, 2.48120352, 4.03750801, 1.9944446 , 1.94577214,
        1.89866677, 2.7033448 , 1.03749316, 1.95866516, 2.61951516,
        1.95616449, 2.68578152, 1.8425

In [66]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Concrete_ml/Quantile Regression/XGBoost.xlsx")

The empirical coverage before calibration is: 75.00%
The empirical coverage after calibration is: 87.50%


In [67]:
plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Concrete_ml/Quantile Regression/XGBoost.xlsx")

# **Conformal Predictions with GPBoost**

In [68]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.1
[GPBoost] [Warning] lambda_l2 is set with lambda=0.1, reg_lambda=0.0 will be ign

In [69]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/GPBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 74.74%


In [70]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/GPBoost.xlsx")

In [71]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Concrete_ml/Quantile Regression/GPBoost.xlsx", model_name='GPBoost')

{'empirical_quantiles': [0.09473684210526316,
  0.15789473684210525,
  0.25263157894736843,
  0.28421052631578947,
  0.4,
  0.47368421052631576,
  0.5368421052631579,
  0.6105263157894737,
  0.6842105263157895,
  0.7578947368421053,
  0.7789473684210526,
  0.8315789473684211,
  0.8421052631578947],
 'average_interval_lengths': [3.696200740089135,
  6.971478150428514,
  11.452476503758005],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 2.4876774494961214,
 'average_crps': 1.753651904247189,
 'nll_values': array([2.44493664, 2.27290749, 1.88267566, 2.16317268, 3.02263315,
        2.44774796, 3.95173056, 2.02882093, 2.99036891, 2.25664227,
        1.9740109 , 1.58275641, 1.99367538, 3.05009352, 3.14619838,
        1.96528589, 3.06675815, 1.78008358, 2.45823231, 1.95625749,
        1.93614879, 1.6497282 , 2.14207813, 1.94567949, 1.40007104,
        1.89614304, 2.39028494, 3.70401124, 1.67525114, 2.11485356,
        1.62764772, 3.81258631, 2.27411304, 1.95305141, 2.74501829,
        

In [72]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/GPBoost.xlsx")

The empirical coverage before calibration is: 77.08%
The empirical coverage after calibration is: 87.50%


In [73]:
plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/GPBoost.xlsx")

# **Conformal Predictions with NGBoost**

In [74]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=3.6975 val_loss=0.0000 scale=1.0000 norm=7.9235
[iter 100] loss=1.6082 val_loss=0.0000 scale=1.0000 norm=1.2485
        0.05        0.1       0.15        0.2        0.3        0.4  \
0  32.014592  32.203626  32.331167  32.432531  32.597588  32.738623   
1  35.979519  36.184546  36.322876  36.432817  36.611838  36.764805   
2  32.955072  33.199864  33.365024  33.496288  33.710030  33.892665   
3  35.476835  35.649774  35.766455  35.859189  36.010192  36.139219   
4  44.105096  44.284425  44.405418  44.501580  44.658163  44.791958   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.870445  33.002267  33.143302  33.308359  33.409723  33.537264   
1  36.907780  37.050754  37.203721  37.382742  37.492683  37.631014   
2  34.063370  34.234075  34.416710  34.630452  34.761716  34.926876   
3  36.259817  36.380416  36.509442  36.660446  36.753180  36.869861   
4  44.917013  45.042068  45.175863  45.332446  45.428607  45.549600   

        0.95  Actual

In [75]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/NGBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 41.05%


In [76]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/NGBoost.xlsx")

In [77]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Concrete_ml/Quantile Regression/NGBoost.xlsx", model_name='NGBoost')

{'empirical_quantiles': [0.2631578947368421,
  0.29473684210526313,
  0.3368421052631579,
  0.3894736842105263,
  0.42105263157894735,
  0.4421052631578947,
  0.47368421052631576,
  0.5052631578947369,
  0.5578947368421052,
  0.6210526315789474,
  0.6421052631578947,
  0.6631578947368421,
  0.6736842105263158],
 'average_interval_lengths': [1.2936762621173648,
  1.9699037677389608,
  2.5283441136711993],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 39.01671472878773,
 'average_crps': 1.9688354549000122,
 'nll_values': array([ 2.62198877e+00,  1.64058014e+00,  4.65463000e+00,  3.67879773e+00,
         4.19161647e-01,  8.35785749e-01,  1.04896543e+02,  8.33128889e-01,
         1.44547052e+00,  8.88213541e-01,  1.21899480e+00, -4.99037054e-01,
         1.03921333e+00,  1.11208362e+01,  3.39331891e+01,  1.06449697e+00,
         1.59424322e+01,  1.26133076e+00,  5.43552751e+00,  3.67054727e+00,
         5.39651083e+00,  1.68432450e+00,  1.25174273e+00,  4.53974143e+00,
         1.79

In [78]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/NGBoost.xlsx")

The empirical coverage before calibration is: 35.42%
The empirical coverage after calibration is: 89.58%


In [79]:
plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/NGBoost.xlsx")

# **Conformal Predictions with Gradient Boosting**

In [80]:
best_params = get_best_model_params(best_scores_autosampler, 'GBM')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  26.987813  27.010837  29.075460  30.181331  32.140294  32.947444   
1  29.147961  30.748399  30.831353  31.108044  33.486502  35.379534   
2  32.847400  31.699499  32.874091  33.863181  33.114571  34.733361   
3  29.512034  31.208464  32.054167  32.277194  33.202341  36.143489   
4  37.484923  40.814909  40.458521  42.198951  42.963439  42.582832   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.934292  32.769460  32.887551  34.446779  34.540826  37.064778   
1  36.531834  37.548582  37.485170  36.793624  36.929682  36.603383   
2  34.901948  34.976242  34.973693  34.971491  36.400388  38.228664   
3  36.099727  36.724769  37.048438  36.725469  35.887752  36.939098   
4  43.080084  48.607730  47.971167  51.006369  50.533573  54.432530   

        0.95  Actual  
0  39.450754    34.0  
1  37.010959    36.0  
2  40.270811    36.0  
3  36.283408    35.0  
4  55.297066    44.6  


In [81]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/Gradient Boosting.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 69.47%


In [82]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/Gradient Boosting.xlsx")

In [83]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Concrete_ml/Quantile Regression/Gradient Boosting.xlsx", model_name='GBM')

{'empirical_quantiles': [0.12631578947368421,
  0.17894736842105263,
  0.23157894736842105,
  0.24210526315789474,
  0.30526315789473685,
  0.37894736842105264,
  0.4631578947368421,
  0.5263157894736842,
  0.6105263157894737,
  0.6736842105263158,
  0.7263157894736842,
  0.8,
  0.8210526315789474],
 'average_interval_lengths': [3.3370424334554,
  6.168637183047756,
  9.594840726947018],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 2.6803153348748294,
 'average_crps': 1.8309665487107354,
 'nll_values': array([2.2904652 , 1.81506268, 1.85117095, 1.78356909, 2.64741994,
        1.78827389, 5.24987144, 0.9168704 , 3.18448767, 1.91143995,
        0.82063652, 1.62768746, 1.47888921, 3.74974095, 3.15419644,
        1.76774798, 3.45809858, 1.8620798 , 2.65816747, 1.4195722 ,
        2.21286814, 2.31879014, 1.99509863, 1.72727904, 1.9131722 ,
        1.68718428, 2.43418329, 3.70615132, 1.71131044, 1.73892534,
        1.73969513, 2.93543244, 1.98570215, 1.45300916, 2.71700855,
        1

In [84]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 68.75%
The empirical coverage after calibration is: 89.58%


In [85]:
plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/Gradient Boosting.xlsx")

# **Conformal Predictions with CatBoost**

In [86]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  23.951718  26.043609  26.826630  30.171087  31.488976  32.999250   
1  36.319458  26.916683  30.563453  34.947704  32.578858  35.040837   
2  27.344639  30.072801  31.961591  34.106772  34.195299  33.953233   
3  28.348779  27.043839  30.556531  32.214878  34.115885  34.288673   
4  45.920188  51.385214  50.049757  47.518827  43.530729  43.337281   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.924015  33.094443  33.606858  34.657001  36.964338  38.817069   
1  35.383753  36.557964  37.129522  36.298526  37.022176  37.840420   
2  34.012001  34.947676  35.386587  36.128718  38.488649  40.933528   
3  36.302601  36.971257  35.831914  36.467350  37.080694  38.789735   
4  41.157167  40.459049  37.831832  40.802023  43.104250  45.827130   

        0.95  Actual  
0  41.147929    34.0  
1  38.673735    36.0  
2  45.818787    36.0  
3  39.621778    35.0  
4  43.887287    44.6  


In [87]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/CatBoost.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 75.79%


In [88]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/CatBoost.xlsx")

In [89]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Concrete_ml/Quantile Regression/CatBoost.xlsx", model_name='CatBoost')

{'empirical_quantiles': [0.11578947368421053,
  0.15789473684210525,
  0.17894736842105263,
  0.2736842105263158,
  0.37894736842105264,
  0.4421052631578947,
  0.4631578947368421,
  0.6210526315789474,
  0.6421052631578947,
  0.7052631578947368,
  0.7684210526315789,
  0.8,
  0.8526315789473684],
 'average_interval_lengths': [4.192623340202685,
  8.743891916321235,
  11.703936359904999],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 769163393053.1395,
 'average_crps': 1.9589731363013383,
 'nll_values': array([2.59401412e+00, 9.55121347e-01, 2.70717192e+00, 2.22279880e+00,
        5.92655085e+12, 1.03529441e+00, 4.85924552e+00, 1.50348057e+00,
        3.30039196e+00, 2.44846293e+00, 1.35517133e+00, 1.81724415e+00,
        2.03580809e+00, 2.72391550e+00, 2.88418618e+00, 2.30971981e+00,
        2.79516963e+00, 2.25953848e+00, 2.53823866e+00, 1.78413888e+00,
        2.49160053e+00, 2.26287540e+00, 1.97081079e+00, 2.12482420e+00,
        1.64085810e+00, 2.32185100e+00, 2.41907972e+0

In [90]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/CatBoost.xlsx")

The empirical coverage before calibration is: 77.08%
The empirical coverage after calibration is: 81.25%


In [91]:
plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/CatBoost.xlsx")

# **Conformal Predictions with HistGradientBoosting**

In [99]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0  27.312290  26.278639  27.027295  31.066164  31.561661  32.759390   
1  29.010019  33.091521  32.796920  33.468847  33.893160  36.101501   
2  32.142212  32.290533  33.951386  34.034620  32.628951  34.317240   
3  30.272422  33.300983  33.242230  33.266150  34.032842  35.850313   
4  35.883220  40.546536  42.681392  42.455334  47.365316  43.101168   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  32.868530  32.870693  32.990844  33.221008  33.455008  36.260778   
1  35.733392  37.431513  37.499009  37.334422  36.770650  38.556987   
2  34.955653  34.915780  35.010079  35.520471  35.035302  38.885263   
3  35.821102  36.119598  37.549534  37.001028  37.193366  39.122436   
4  43.372518  46.262387  47.018456  49.858499  48.891879  50.291903   

        0.95  Actual  
0  38.636170    34.0  
1  41.194171    36.0  
2  39.935590    36.0  
3  41.194171    35.0  
4  55.447779    44.6  


In [100]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/HGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 73.68%


In [101]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/HGBM.xlsx")

In [102]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles, "./drive/MyDrive/Concrete_ml/Quantile Regression/HGBM.xlsx", model_name='HGBM')

{'empirical_quantiles': [0.1368421052631579,
  0.15789473684210525,
  0.22105263157894736,
  0.2736842105263158,
  0.37894736842105264,
  0.42105263157894735,
  0.4842105263157895,
  0.5368421052631579,
  0.631578947368421,
  0.6736842105263158,
  0.7052631578947368,
  0.7789473684210526,
  0.8736842105263158],
 'average_interval_lengths': [3.2221584736597317,
  7.318947187709871,
  10.617863631966442],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 2.6405976765288988,
 'average_crps': 1.7829100603334451,
 'nll_values': array([ 2.20907685,  2.23086684,  1.87858228,  2.14948004,  2.72315943,
         1.46567942,  3.55260556,  2.38241639,  3.4327986 ,  1.77056811,
         1.8813746 ,  1.1626407 ,  2.41412899,  3.49236037,  3.15596517,
         1.69693932,  3.03909511,  1.84329951,  2.45242698,  2.05162611,
         1.96880946,  1.70109717,  2.82612003,  2.08537364,  1.43651772,
         2.11256862,  2.09605004,  3.57906047,  1.77371954,  2.41585859,
         1.41925277,  4.4188962

In [103]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Concrete_ml/Quantile Regression/HGBM.xlsx")

The empirical coverage before calibration is: 70.83%
The empirical coverage after calibration is: 85.42%


In [104]:
plot_quantile_intervals_to_excel(predictions_HGBR_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/HGBM.xlsx")

# **Conformal Predictions with PGBM**

In [105]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [106]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/500, Train metric: 9.2614
Estimator 1/500, Train metric: 8.6107
Estimator 2/500, Train metric: 8.1920
Estimator 3/500, Train metric: 7.6708
Estimator 4/500, Train metric: 7.4130
Estimator 5/500, Train metric: 7.0545
Estimator 6/500, Train metric: 6.7347
Estimator 7/500, Train metric: 6.3296
Estimator 8/500, Train metric: 6.0250
Estimator 9/500, Train metric: 5.7801
Estimator 10/500, Train metric: 5.5615
Estimator 11/500, Train metric: 5.2922
Estimator 12/500, Train metric: 5.0620
Estimator 13/500, Train metric: 4.9032
Estimator 14/500, Train metric: 4.7388
Estimator 15/500, Train metric: 4.5443
Estimator 16/500, Train metric: 4.3841
Estimator 17/500, Train metric: 4.2440
Estimator 18/500, Train metric: 4.1093
Estimator 19/500, Train metric: 3.9801
Estimator 20/500, Train metric: 3.8436
Estimator 21/500, Train metric: 3.7578
Estimator 22/500, Train metric: 3.6645
Estimator 23/500, Train metric: 3.5425
Estimator 24/500, Train metric: 3.4379
Estimator 25/500, T

In [107]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/PGBM.xlsx")

First few actual values: [34.  36.  36.  35.  44.6]
Coverage of 90% prediction interval: 4.21%


In [108]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Concrete_ml/Quantile Regression/PGBM.xlsx")

In [109]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Concrete_ml/Quantile Regression/PGBM.xlsx", model_name='PGBM')

{'empirical_quantiles': [0.43157894736842106,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.45263157894736844,
  0.4631578947368421,
  0.47368421052631576,
  0.47368421052631576],
 'average_interval_lengths': [0.17247888263903172,
  0.263222266749332,
  0.3324672518278426],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': 368.538529735143,
 'average_crps': 1.7705132004184774,
 'nll_values': array([ 7.04882843e+01,  2.38766909e+01,  1.05735064e+02,  1.27248537e+02,
         5.02643601e+01,  9.27671182e-01,  9.08643750e+02,  1.67115204e+01,
         5.26642360e+00, -7.18775827e-01, -6.40041629e-01,  1.74080018e-01,
         1.06773530e+02,  3.61842755e+02,  1.00007493e+03,  2.65376874e+01,
         9.60840068e+02, -2.68565054e-01,  2.35252944e+02,  3.53182407e+01,
         3.67429662e+02,  9.08655849e+01,  1.98735031e+01,  1.99273614e+02,
     

In [110]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Concrete_ml/Quantile Regression/PGBM.xlsx")

The empirical coverage before calibration is: 2.08%
The empirical coverage after calibration is: 87.50%


In [112]:
plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Concrete_ml/Quantile Regression/PGBM.xlsx")